**user_message:**

I have uploaded the following files for you to use: hr_raw.csv

I uploaded the file: `hr_raw.csv`

"I have a dataset with columns: [Employee_ID, Full_Name, Department, Job_Title, Hire_Date, Performance_Rating, Experience_Years, Status, Work_Mode, Salary, Country, Age, Job_Level].
Task: Write Python code using Pandas to:
Calculate the average salary per Country and Department.
Find the correlation between Experience_Years and Salary.
Create a pivot table showing the count of employees by Performance_Rating and Work_Mode.
Identify the top 5 highest-paid Job Titles in each country.
Constraint: Use efficient code to handle 2 million rows."

In [ ]:
# Load the HR CSV with memory-conscious dtypes and show a quick head for validation
import pandas as pd
from tqdm import tqdm

dtype_map = {
    'Employee_ID': 'string',
    'Full_Name': 'string',
    'Department': 'category',
    'Job_Title': 'category',
    'Performance_Rating': 'category',
    'Status': 'category',
    'Work_Mode': 'category',
    'Country': 'category',
    'Job_Level': 'category',
    'Age': 'Int16',
    'Experience_Years': 'Int16',
    'Salary': 'float32'
}

df_hr = pd.read_csv('hr_raw.csv', dtype=dtype_map, parse_dates=['Hire_Date'])
print(df_hr.head())

In [ ]:
# Compute all requested outputs efficiently from the in-memory dataframe and print compact results
import pandas as pd
from tqdm import tqdm

avg_salary_country_dept = df_hr.groupby(['Country', 'Department'], observed=True, sort=False)['Salary'].mean().reset_index()
correlation_exp_salary = df_hr[['Experience_Years', 'Salary']].dropna().corr(method='pearson').loc['Experience_Years', 'Salary']
pivot_perf_workmode = pd.crosstab(df_hr['Performance_Rating'], df_hr['Work_Mode'], dropna=False)
job_title_country_avg = df_hr.groupby(['Country', 'Job_Title'], observed=True, sort=False)['Salary'].mean().reset_index()
top_5_job_titles_each_country = job_title_country_avg.sort_values(['Country', 'Salary'], ascending=[True, False]).groupby('Country', observed=True).head(5)

print(avg_salary_country_dept.head(10))
print(correlation_exp_salary)
print(pivot_perf_workmode)
print(top_5_job_titles_each_country.head(20))

**message:**

### Efficient Pandas solution for a 2M-row HR dataset

Here’s a memory-conscious Pandas approach that does exactly what you asked. I used categorical dtypes for repeated string columns and vectorized groupby/crosstab operations, which is the right pattern for datasets on the order of *millions* of rows.

```python
import pandas as pd

# Memory-efficient load for large CSVs
dtype_map = {
    'Employee_ID': 'string',
    'Full_Name': 'string',
    'Department': 'category',
    'Job_Title': 'category',
    'Performance_Rating': 'category',
    'Status': 'category',
    'Work_Mode': 'category',
    'Country': 'category',
    'Job_Level': 'category',
    'Age': 'Int16',
    'Experience_Years': 'Int16',
    'Salary': 'float32'
}

df = pd.read_csv(
    'hr_raw.csv',
    dtype=dtype_map,
    parse_dates=['Hire_Date']
)

# Keep only columns needed for this task to reduce memory pressure
df = df[
    [
        'Country',
        'Department',
        'Job_Title',
        'Performance_Rating',
        'Work_Mode',
        'Experience_Years',
        'Salary'
    ]
]

# 1. Average salary per Country and Department
avg_salary_per_country_department = (
    df.groupby(['Country', 'Department'], observed=True)['Salary']
      .mean()
      .reset_index(name='Average_Salary')
)

# 2. Correlation between Experience_Years and Salary
experience_salary_correlation = (
    df[['Experience_Years', 'Salary']]
    .dropna()
    .corr(method='pearson')
    .loc['Experience_Years', 'Salary']
)

# 3. Pivot table: count of employees by Performance_Rating and Work_Mode
employee_count_pivot = pd.crosstab(
    df['Performance_Rating'],
    df['Work_Mode'],
    dropna=False
)

# 4. Top 5 highest-paid Job Titles in each country
# Using average salary per Job Title within each country
job_title_salary_by_country = (
    df.groupby(['Country', 'Job_Title'], observed=True)['Salary']
      .mean()
      .reset_index(name='Average_Salary')
)

top_5_highest_paid_job_titles_each_country = (
    job_title_salary_by_country
    .sort_values(['Country', 'Average_Salary'], ascending=[True, False])
    .groupby('Country', observed=True)
    .head(5)
    .reset_index(drop=True)
)

# Output
print(avg_salary_per_country_department.head(10))
print(experience_salary_correlation)
print(employee_count_pivot)
print(top_5_highest_paid_job_titles_each_country.head(20))
```

### Notes on efficiency

For a 2 million row file, the main performance wins are using `category` for columns like `Country`, `Department`, `Job_Title`, `Performance_Rating`, and `Work_Mode`, trimming the dataframe to only needed columns, and using `groupby(..., observed=True)` to avoid unnecessary category combinations.

### Assumption I used

For *top 5 highest-paid Job Titles in each country*, I interpreted that as the **top 5 job titles by average salary within each country**, which is usually the most useful business definition. If you want, I can also give you the version for the **top 5 individual employees per country** instead.

In [ ]:
# Build a compact HR analysis dataset from the in-memory dataframe and print key summaries for dashboard design
import pandas as pd
from tqdm import tqdm

analysis_df = df_hr.copy()
analysis_df['Salary'] = pd.to_numeric(analysis_df['Salary'], errors='coerce')
analysis_df['Experience_Years'] = pd.to_numeric(analysis_df['Experience_Years'], errors='coerce')
analysis_df['Age'] = pd.to_numeric(analysis_df['Age'], errors='coerce')

country_dept_salary = analysis_df.groupby(['Country', 'Department'], observed=True)['Salary'].mean().reset_index()
country_headcount = analysis_df.groupby('Country', observed=True).size().reset_index(name='Employees').sort_values('Employees', ascending=False)
dept_headcount = analysis_df.groupby('Department', observed=True).size().reset_index(name='Employees').sort_values('Employees', ascending=False)
perf_workmode = pd.crosstab(analysis_df['Performance_Rating'], analysis_df['Work_Mode']).reset_index()
job_title_country_avg = analysis_df.groupby(['Country', 'Job_Title'], observed=True)['Salary'].mean().reset_index(name='Average_Salary')
top_jobs_country = job_title_country_avg.sort_values(['Country', 'Average_Salary'], ascending=[True, False]).groupby('Country', observed=True).head(5)
correlation_val = analysis_df[['Experience_Years', 'Salary']].dropna().corr().loc['Experience_Years', 'Salary']

print(analysis_df.head())
print(country_headcount.head(10))
print(dept_headcount.head(10))
print(correlation_val)
print(top_jobs_country.head(15))

In [ ]:
# Prepare compact tables for an HR dashboard from the existing dataframe and save a reusable summary workbook
import pandas as pd
from tqdm import tqdm

kpis_df = pd.DataFrame([
    {'Metric': 'Employees', 'Value': int(len(analysis_df))},
    {'Metric': 'Avg Salary', 'Value': float(analysis_df['Salary'].mean())},
    {'Metric': 'Median Salary', 'Value': float(analysis_df['Salary'].median())},
    {'Metric': 'Avg Experience', 'Value': float(analysis_df['Experience_Years'].mean())},
    {'Metric': 'Correlation Exp-Salary', 'Value': float(correlation_val)}
])

salary_by_country = analysis_df.groupby('Country', observed=True)['Salary'].mean().reset_index(name='Average_Salary').sort_values('Average_Salary', ascending=False)
salary_by_department = analysis_df.groupby('Department', observed=True)['Salary'].mean().reset_index(name='Average_Salary').sort_values('Average_Salary', ascending=False)
perf_counts = analysis_df['Performance_Rating'].value_counts(dropna=False).rename_axis('Performance_Rating').reset_index(name='Employees')
workmode_counts = analysis_df['Work_Mode'].value_counts(dropna=False).rename_axis('Work_Mode').reset_index(name='Employees')

with pd.ExcelWriter('hr_dashboard_summary.xlsx', engine='openpyxl') as writer:
    kpis_df.to_excel(writer, sheet_name='KPIs', index=False)
    country_headcount.to_excel(writer, sheet_name='Country Headcount', index=False)
    dept_headcount.to_excel(writer, sheet_name='Dept Headcount', index=False)
    salary_by_country.to_excel(writer, sheet_name='Salary by Country', index=False)
    salary_by_department.to_excel(writer, sheet_name='Salary by Dept', index=False)
    perf_workmode.to_excel(writer, sheet_name='Perf x WorkMode', index=False)
    top_jobs_country.to_excel(writer, sheet_name='Top Jobs by Country', index=False)

print(kpis_df)
print(salary_by_country)
print(salary_by_department)
print(perf_counts)
print(workmode_counts)

In [ ]:
import pandas as pd
from tqdm import tqdm

scatter_sample = analysis_df[['Experience_Years', 'Salary']].dropna().sample(n=1000, random_state=42)
country_dept_salary_top = country_dept_salary.sort_values('Salary', ascending=False).head(20)
print(scatter_sample.head())
print(country_dept_salary_top.head(10))

**message:**

### HR analysis and dashboard

I went ahead and analyzed the full `hr_raw.csv` dataset in memory and built an interactive dashboard view from it.

#### What stands out

This dataset has **2,000,000 employees**, so the summaries are meaningful at scale.

The clearest signal is the relationship between *experience* and *salary*. The Pearson correlation between `Experience_Years` and `Salary` is **0.924**, which is a **very strong positive relationship**. In plain English, salary rises very consistently with experience in this workforce.

Compensation is also pretty structured by function. **IT** is the highest-paid department on average at about **96.3k**, followed by **Finance** at **94.2k**, then **Sales** at **91.2k**. **Operations** and especially **HR** sit lower.

From a workforce mix perspective, the biggest countries by headcount are **United Kingdom**, **Germany**, and **France**. The largest department is **Sales**, followed by **IT** and **Operations**.

Performance distribution is concentrated in the middle. Most employees are rated **Good**, then **Satisfactory**, with fewer in **Excellent** and **Needs Improvement**. Across work arrangements, **On-site** is the dominant mode, then **Remote**, then **Hybrid**.

For high-paying roles, the top titles across countries are heavily concentrated in technical jobs like **DevOps Engineer**, **System Admin**, **Data Engineer**, **Software Developer**, and **IT Manager**.

#### KPI snapshot

Here are the main numbers I computed:

- **Employees:** 2,000,000  
- **Average salary:** 89,948  
- **Median salary:** 80,829  
- **Average experience:** 6.27 years  
- **Experience-salary correlation:** 0.924  

#### Interactive dashboard charts

[JULIUS_CHART_CONFIG]: 8adc4306-429b-4c6e-a542-6802b8456aa3

[JULIUS_CHART_CONFIG]: 90ff8162-ad3f-4d13-84d1-fdb2fcb6427a

[JULIUS_CHART_CONFIG]: 21c3a284-5d58-4d3f-b200-ab2709d6c003

[JULIUS_CHART_CONFIG]: fdad899c-a0b9-4345-b20e-4a2b72911e6f

#### Exported summary file

I also saved the analysis outputs into an Excel workbook so you can download and inspect the tables directly:


It includes:
- `KPIs`
- `Country Headcount`
- `Dept Headcount`
- `Salary by Country`
- `Salary by Dept`
- `Perf x WorkMode`
- `Top Jobs by Country`

#### Quick interpretation

If this were a real HR dashboard, I’d summarize it like this:

The organization appears to have a **strongly experience-driven pay structure**, with fairly consistent salary progression. The **highest compensation clusters in IT and Finance**, while **HR is materially lower-paid** on average. The workforce is concentrated geographically in the **UK, Germany, and France**, and operationally in **Sales and IT**. Performance ratings are centered around **Good**, which may indicate either a healthy workforce or a rating system with central tendency.

If you want, I can do the next step and turn this into a **single polished executive dashboard** with:
- KPI cards
- a country salary heatmap
- top job titles by country table
- department breakdowns
- optional filters for country, department, and work mode

In [ ]:
# Quantify why IT leads salaries by decomposing salary across experience, level, country, and top titles using the in-memory HR dataframe
import pandas as pd
from tqdm import tqdm

it_df = analysis_df.copy()
it_df['Salary'] = pd.to_numeric(it_df['Salary'], errors='coerce')
it_df['Experience_Years'] = pd.to_numeric(it_df['Experience_Years'], errors='coerce')
it_df['Age'] = pd.to_numeric(it_df['Age'], errors='coerce')

avg_by_dept = it_df.groupby('Department', observed=True).agg(
    avg_salary=('Salary', 'mean'),
    avg_experience=('Experience_Years', 'mean'),
    avg_age=('Age', 'mean'),
    employees=('Employee_ID', 'count')
).reset_index().sort_values('avg_salary', ascending=False)

level_salary = it_df.groupby(['Department', 'Job_Level'], observed=True)['Salary'].mean().reset_index()
level_mix = it_df.groupby(['Department', 'Job_Level'], observed=True).size().reset_index(name='employees')
country_it_salary = it_df[it_df['Department'] == 'IT'].groupby('Country', observed=True)['Salary'].mean().reset_index(name='avg_salary')
top_it_titles = it_df[it_df['Department'] == 'IT'].groupby('Job_Title', observed=True).agg(avg_salary=('Salary', 'mean'), employees=('Employee_ID', 'count')).reset_index().sort_values('avg_salary', ascending=False)

print(avg_by_dept)
print(level_salary.head(15))
print(level_mix.head(15))
print(country_it_salary)
print(top_it_titles.head(10))

**message:**

### Why IT leads salaries in this dataset

Short version: **IT is not leading because IT employees are much older or much more experienced overall**. The averages are almost the same across departments. IT leads because the **pay ladder for IT roles is higher at nearly every job level**, and the department is filled with **higher-paying technical titles**.

#### 1. It’s **not** mainly an experience effect

If IT were highest-paid simply because IT workers had more tenure, you’d expect IT to have much higher average experience. It doesn’t.

Across departments:

- **IT** average salary: **96.3k**, average experience: **6.27 years**
- **Finance** average salary: **94.2k**, average experience: **6.27 years**
- **Sales** average salary: **91.2k**, average experience: **6.29 years**
- **Operations** average salary: **84.7k**, average experience: **6.28 years**
- **HR** average salary: **74.4k**, average experience: **6.26 years**

So the department salary gap is happening even though average experience is basically flat.

#### 2. IT pays more **within level**

This is the strongest explanation in your data.

By `Job_Level`:

- **Junior**
  - IT: **55.6k**
  - Finance: **52.5k**
  - HR: **42.9k**
- **Mid**
  - IT: **92.6k**
  - Finance: **89.9k**
  - HR: **68.7k**
- **Senior**
  - IT: **148.8k**
  - Finance: **147.7k**
  - HR: **117.5k**
- **Director**
  - IT: **233.4k**
  - Finance: **239.9k**
  - HR: **194.2k**

So IT beats most departments at **junior, mid, and senior** levels. Finance slightly exceeds IT at director level, but IT still wins overall because its structure is strong across the full ladder.

#### 3. IT roles themselves are clustered in high-paying titles

Top-paying IT titles by average salary are:

- **DevOps Engineer**: **96.6k**
- **System Admin**: **96.6k**
- **Data Engineer**: **96.3k**
- **IT Manager**: **96.2k**
- **Software Developer**: **95.9k**

Those are all relatively high-value technical roles. In most organizations, roles tied to infrastructure, software delivery, systems reliability, and data tend to command stronger pay than administrative or support-heavy functions.

#### 4. IT’s salary leadership is consistent across countries

Average IT salary is tightly clustered across countries:

- France: **96.1k**
- Germany: **96.3k**
- Italy: **96.5k**
- Netherlands: **96.3k**
- Poland: **96.2k**
- Spain: **96.6k**
- United Kingdom: **96.2k**

That consistency suggests this is a **structural pay premium for IT**, not a country-mix artifact.

### Bottom line

In *this dataset*, IT leads salaries because:

- **experience is similar across departments**
- **IT salary bands are higher at most job levels**
- **IT contains high-paying technical roles**
- **the premium shows up consistently across countries**

So the cleanest conclusion is: **IT is paid more because the job family itself is priced higher, not because IT employees are unusually senior on average.**

If you want, I can take this one step further and quantify it with a **decomposition model** showing how much of the IT salary premium comes from:
- job level mix
- country mix
- experience
- title mix

In [ ]:
# Compute Gini coefficient and a few supporting metrics from the in-memory HR dataframe for the README
import numpy as np
import pandas as pd
from tqdm import tqdm

salary_vals = pd.to_numeric(analysis_df['Salary'], errors='coerce').dropna().to_numpy(dtype=float)
salary_vals = salary_vals[salary_vals >= 0]
salary_sorted = np.sort(salary_vals)
num_vals = salary_sorted.size
index_vals = np.arange(1, num_vals + 1)
gini_coef = (2 * np.sum(index_vals * salary_sorted) / (num_vals * np.sum(salary_sorted))) - ((num_vals + 1) / num_vals)

summary_metrics = {
    'employees': int(len(analysis_df)),
    'avg_salary': float(pd.to_numeric(analysis_df['Salary'], errors='coerce').mean()),
    'median_salary': float(pd.to_numeric(analysis_df['Salary'], errors='coerce').median()),
    'salary_gini': float(gini_coef),
    'exp_salary_corr': float(correlation_val)
}

print(summary_metrics)

**message:**

## README.md

# HR Analytics Dashboard

## Business Problem

Organizations often have HR data, but not real visibility into what actually drives compensation, workforce structure, and performance patterns at scale.

This project analyzes a large HR dataset with **2,000,000 employee records** to answer practical business questions such as:

- Which countries and departments have the highest average salaries
- Whether experience actually explains salary differences
- How performance ratings vary by work mode
- Which job titles command the highest pay in each country
- How equally or unequally income is distributed across the workforce

The goal is to turn raw HR records into a decision-ready dashboard that helps HR leaders, finance teams, and executives understand **pay structure, workforce composition, and inequality in compensation**.

---

## Tools Used

This project was built with **Python** and a lightweight analytics stack focused on performance for large datasets.

- **Pandas** for data cleaning, grouping, pivoting, and large-scale aggregation
- **NumPy** for efficient numerical computation
- **Jupyter Notebook** for interactive analysis
- **Interactive charting** for dashboard-style visual exploration
- **Excel export** for sharing summary tables with non-technical stakeholders

To keep the workflow efficient for millions of rows, repeated text columns were handled with **categorical dtypes**, and aggregations were written using vectorized `groupby`, `crosstab`, and sorting operations.

---

## The Wow Factor

The biggest surprise in the analysis is how strongly salary tracks experience.

The correlation between `Experience_Years` and `Salary` is:

**0.924**

That is an extremely strong positive relationship. It means compensation in this dataset is not random or weakly tied to tenure. Instead, salary appears to follow a very structured progression as experience increases.

Another strong finding is that **IT leads all departments in average salary**, not because IT employees are drastically older or more experienced, but because the **pay bands for IT roles are higher across job levels** and the department is full of high-value technical roles like:

- DevOps Engineer
- System Admin
- Data Engineer
- IT Manager
- Software Developer

This makes the dashboard useful not just descriptively, but strategically. It helps explain *why* compensation differences exist.

---

## Visualization

The dashboard includes visuals designed to make large-scale HR patterns immediately understandable.

Typical views include:

- **Employees by Country** to show workforce concentration
- **Average Salary by Department** to compare pay structures
- **Performance Rating by Work Mode** to examine workforce outcomes
- **Salary vs Experience** scatter plot to reveal the salary ladder
- **Top-paying Job Titles by Country** to highlight role-level compensation leaders

These visuals turn millions of rows into a format that executives can scan in seconds.

---

## Economic Insight

This dataset reflects a structured labor market rather than a chaotic one.

A few high-level economic patterns stand out:

**First**, compensation is highly tied to human capital. Experience has a very strong relationship with pay, suggesting a mature salary ladder.

**Second**, departments are priced differently based on market value of skills. IT and Finance sit at the top, while HR and Operations are lower on average.

**Third**, geography matters for workforce size more than for IT pay. Countries differ meaningfully in headcount, but IT salaries remain relatively consistent across countries, which suggests that technical roles carry a cross-market premium.

This is the kind of insight that can support compensation planning, org design, hiring strategy, and internal equity reviews.

---

## Income Distribution: Gini Coefficient

To measure salary inequality, this project calculates the **Gini Coefficient** on employee salaries.

### Result

**Gini Coefficient = 0.269**

### Interpretation

A Gini coefficient of:

- `0` means perfect equality
- `1` means perfect inequality

So a value of **0.269** suggests **moderate income inequality** in the workforce.

That means salaries are not evenly distributed, but they are also not extremely concentrated among a very small group. This fits the broader story in the data:

- there is a strong experience-driven pay ladder
- higher-paying departments like IT and Finance pull the upper end upward
- job level and title create structured differences in earnings

In other words, the inequality here appears to be **systematic and role-driven**, not purely arbitrary.

---

## Key Results

### Workforce scale
**2,000,000 employees**

### Average salary
**89,948**

### Median salary
**80,829**

### Experience-salary correlation
**0.924**

### Salary inequality
**Gini = 0.269**

### Highest-paying department
**IT**

---

## Project Outcome

This project transforms raw HR data into a scalable analytics workflow and executive-ready dashboard.

It helps answer:

- where the workforce is concentrated
- which departments are most expensive
- what drives salary differences
- how performance and work mode interact
- whether income distribution is relatively equal or unequal

That makes it valuable for **HR analytics**, **workforce planning**, **compensation benchmarking**, and **leadership reporting**.

---

If you want, I can also turn this into:
- a **cleaner polished GitHub README with badges**
- a **full project README with installation and usage sections**
- or a **README.md file export** you can download directly